# GA1-220501094-01-AA1-EV06: Limpieza, análisis y estructuración de datos con Pandas (Instacart)

**Programa de Formación:** Tecnólogo en Análisis y Desarrollo de Software (ADSO)  
**Competencia:** 220501094 - Desarrollar la solución de analítica de datos según requisitos del negocio  
**Resultados de Aprendizaje:**  
- Aplicar técnicas de limpieza y transformación de datos tabulares con Pandas.  
- Explorar patrones de consumo y hábitos de compra mediante filtrado, ordenamiento y agregaciones.  
- Integrar múltiples conjuntos de datos relacionales en una estructura analítica maestra validada.

---

### Contenido del Cuaderno:
1. **Configuración y Preparación del Entorno**
2. **Lección 2: Limpieza y Estandarización de Datos Crudos**
   - *Laboratorio Guiado 2:* Pipeline de limpieza con informe de cambios y validación con `assert`.
   - *Reto 2:* 7 ejercicios resueltos sobre duplicados, integridad, imputación y optimización.
3. **Lección 3: Filtrado, Ordenamiento y Agrupación de Datos**
   - *Laboratorio Guiado 3:* Perfil de hábitos de compra, segmentación con `pd.cut` y cruce por franjas horarias.
   - *Reto 3:* 7 ejercicios resueltos sobre ventanas temporales, horas pico, tamaños de cesta y recompra.
4. **Lección 4: Combinación y Estructuración de Múltiples Datasets**
   - *Laboratorio Guiado 4:* Ficha analítica por producto, ranking departamental y persistencia en Parquet/CSV.
   - *Reto 4:* 7 ejercicios resueltos con tabla maestra, agregaciones multivariable, tablas dinámicas y análisis cruzados.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuración dinámica de rutas para compatibilidad tanto desde notebooks/ como desde la raíz del proyecto
if Path('data/raw').exists():
    RUTA_RAW = Path('data/raw')
    RUTA_PROCESSED = Path('data/processed')
elif Path('../data/raw').exists():
    RUTA_RAW = Path('../data/raw')
    RUTA_PROCESSED = Path('../data/processed')
else:
    RUTA_RAW = Path(r'C:/Users/Admin/.gemini/antigravity/scratch/proyecto-ciencia-datos/data/raw')
    RUTA_PROCESSED = Path(r'C:/Users/Admin/.gemini/antigravity/scratch/proyecto-ciencia-datos/data/processed')

RUTA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Opciones de visualización de pandas
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

print("Entorno inicializado correctamente.")
print(f"Ruta de datos crudos: {RUTA_RAW.resolve()}")
print(f"Ruta de datos procesados: {RUTA_PROCESSED.resolve()}")


Entorno inicializado correctamente.
Ruta de datos crudos: C:\Users\Admin\.gemini\antigravity\scratch\proyecto-ciencia-datos\data\raw
Ruta de datos procesados: C:\Users\Admin\.gemini\antigravity\scratch\proyecto-ciencia-datos\data\processed


---
## Lección 2: Limpieza y estandarización de datos crudos

En esta lección se resuelven las anomalías encontradas en la fase de exploración:
- Detección y eliminación de duplicados exactos en pedidos.
- Optimización de tipos de datos (`int32`, `int16`, `int8`, `Int16`) para reducir el uso de memoria.
- Limpieza y estandarización de textos en nombres de productos (manejo de nulos, caracteres invisibles, espacios múltiples y formato tipo título).
- Validación de integridad con pruebas automáticas `assert`.

### Laboratorio Guiado 2: Pipeline de limpieza con informe de cambios


#### Paso 1: Recargar los datos crudos
Cargamos los archivos CSV delimitados por punto y coma (`;`).


In [2]:
orders_crudo   = pd.read_csv(RUTA_RAW / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA_RAW / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA_RAW / 'order_products.csv', sep=';')

print("Dimensiones crudas:")
print(f"orders:       {orders_crudo.shape}")
print(f"products:     {products_crudo.shape}")
print(f"order_prods:  {op_crudo.shape}")


Dimensiones crudas:
orders:       (478967, 6)
products:     (49694, 4)
order_prods:  (4545007, 4)


#### Paso 2: Función que anota el estado de la tabla
Definimos la función `anotar` para registrar una métrica del estado de la tabla (filas, nulos y memoria en MB) tras cada transformación.


In [3]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

print("Función 'anotar' definida exitosamente.")


Función 'anotar' definida exitosamente.


#### Paso 3: Funciones modulares de limpieza
Cada función implementa una transformación atómica y devuelve un DataFrame.


In [4]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({
        'order_id': 'int32',
        'user_id': 'int32',
        'order_number': 'int16',
        'order_dow': 'int8',
        'order_hour_of_day': 'int8',
        'days_since_prior_order': 'Int16'
    })

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({
        'order_id': 'int32',
        'product_id': 'int32',
        'add_to_cart_order': 'Int16',
        'reordered': 'int8'
    })

print("Funciones de limpieza modulares definidas.")


Funciones de limpieza modulares definidas.


#### Paso 4: Encadenar los pasos con `.pipe()`
Ejecutamos el pipeline secuencial registrando el estado intermedio y final.


In [5]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos corregidos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres limpios'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos corregidos'))

print("Pipeline completado para todas las tablas.")


Pipeline completado para todas las tablas.


#### Paso 5: Informe de cambios
Consolidamos el informe tabular que detalla el impacto cuantitativo del pipeline de limpieza.


In [6]:
informe = pd.DataFrame(registro)
informe


         tabla                paso    filas  nulos  memoria_mb
0       orders             0 crudo   478967  28819        23.0
1       orders    1 sin duplicados   478952  28817        23.0
2       orders  2 tipos corregidos   478952  28817         7.2
3     products             0 crudo    49694   1258         3.1
4     products   1 nombres limpios    49694      0         3.1
5  order_prods             0 crudo  4545007    836       145.4
6  order_prods  1 tipos corregidos  4545007    836        54.5


**Interpretación del informe de cambios:**
1. **Eliminación de duplicados:** Se removieron exactamente 15 pedidos repetidos en `orders` (de 478,967 a 478,952 filas) y 2 valores nulos asociados (pedidos que eran primeros pedidos repetidos).
2. **Optimización de memoria:** La corrección de tipos redujo el consumo de `orders` de 23.0 MB a 7.2 MB (68.7% de ahorro) y el de `order_products` de 145.4 MB a 54.5 MB (62.5% de ahorro).
3. **Imputación de nombres:** En `products` se eliminaron los 1,258 nulos imputándolos a `'Unknown'`.
4. **Preservación controlada de nulos:** Los 836 nulos de `add_to_cart_order` en `order_products` se preservaron deliberadamente utilizando el tipo nullable `Int16`.


#### Paso 6: Validación de integridad con `assert`
Comprobamos que las tablas resultantes cumplan rigurosamente con las reglas de negocio.


In [7]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_id'].is_unique, 'product_id repetido'
    assert not p['product_name'].isna().any(), 'hay nombres nulos'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores invalidos'
    print("Validación superada con éxito: todas las aserciones de calidad se cumplen.")

validar(orders_limpio, products_limpio, op_limpio)


Validación superada con éxito: todas las aserciones de calidad se cumplen.


### Reto 2: Ejercicios de Limpieza y Estandarización


#### Ejercicio 2.1: Duplicados en `order_products` e integridad de `reordered`
*Pregunta:* Comprueba si `order_prods` tiene filas duplicadas exactas, si se repite algún par `order_id + product_id` y si `reordered` solo contiene 0 y 1.


In [8]:
dup_exactos = op_limpio.duplicated().sum()
dup_pares = op_limpio.duplicated(subset=['order_id', 'product_id']).sum()
valores_reordered = op_limpio['reordered'].unique()

print(f"Filas duplicadas exactas en order_prods: {dup_exactos}")
print(f"Pares order_id + product_id duplicados:   {dup_pares}")
print(f"Valores únicos en columna reordered:      {valores_reordered}")


Filas duplicadas exactas en order_prods: 0
Pares order_id + product_id duplicados:   0
Valores únicos en columna reordered:      [0 1]


*Interpretación:* La tabla `order_products` no posee duplicados exactos ni pares orden-producto repetidos, garantizando unicidad en cada ítem del carrito. La columna `reordered` es estrictamente binaria (0 y 1).


#### Ejercicio 2.2: Verificación de duplicados y espacios en `aisles` y `departments`
*Pregunta:* Comprueba si `aisles` y `departments` tienen nombres duplicados o con espacios al inicio o al final. ¿Necesitan limpieza de texto?


In [9]:
aisles = pd.read_csv(RUTA_RAW / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA_RAW / 'departments.csv', sep=';')

aisles_dup = aisles['aisle'].duplicated().sum()
aisles_espacios = (aisles['aisle'].str.startswith(' ') | aisles['aisle'].str.endswith(' ')).sum()

dept_dup = departments['department'].duplicated().sum()
dept_espacios = (departments['department'].str.startswith(' ') | departments['department'].str.endswith(' ')).sum()

print(f"Aisles:      {len(aisles)} registros | Duplicados: {aisles_dup} | Con espacios: {aisles_espacios}")
print(f"Departments: {len(departments)} registros  | Duplicados: {dept_dup} | Con espacios: {dept_espacios}")


Aisles:      134 registros | Duplicados: 0 | Con espacios: 0
Departments: 21 registros  | Duplicados: 0 | Con espacios: 0


*Interpretación:* Ninguno de los dos catálogos presenta valores duplicados ni espacios sobrantes al inicio o al final. Por lo tanto, no requieren modificaciones de texto adicionales.


#### Ejercicio 2.3: Investigación de los 836 nulos en `add_to_cart_order`
*Pregunta:* Investiga los 836 nulos de `add_to_cart_order`: ¿en cuántos pedidos ocurren y cuántos productos tiene, como mínimo, cada uno? ¿Qué los explica?


In [10]:
nulos_cart = op_crudo[op_crudo['add_to_cart_order'].isna()]
pedidos_con_nulos = nulos_cart['order_id'].unique()
cant_pedidos_nulos = len(pedidos_con_nulos)

prods_por_pedido_afectado = op_crudo[op_crudo['order_id'].isin(pedidos_con_nulos)].groupby('order_id').size()

print(f"Cantidad de pedidos con nulos en add_to_cart_order: {cant_pedidos_nulos}")
print(f"Mínimo de productos en un pedido afectado:           {prods_por_pedido_afectado.min()}")
print(f"Máximo de productos en un pedido afectado:           {prods_por_pedido_afectado.max()}")
print(f"Promedio de productos en pedidos afectados:          {prods_por_pedido_afectado.mean():.1f}")


Cantidad de pedidos con nulos en add_to_cart_order: 70
Mínimo de productos en un pedido afectado:           65
Máximo de productos en un pedido afectado:           127
Promedio de productos en pedidos afectados:          75.9


*Interpretación:* Los 836 nulos ocurren en exactamente 70 pedidos. Cada uno de estos pedidos tiene como mínimo 65 productos (y hasta 127). Esto se debe a un truncamiento transaccional en el sistema de captura original, el cual solo registraba las posiciones del carrito del 1 al 64.


#### Ejercicio 2.4: Estandarización de nombres en `products.csv`
*Pregunta:* Carga de nuevo `products.csv` en una variable `products_crudo` y cuenta cuántos nombres (no nulos) cambiaron con la estandarización. Muestra tres que hayan cambiado.


In [11]:
prod_no_nulos = products_crudo[products_crudo['product_name'].notna()].copy()
prod_estandarizado = limpiar_nombres(prod_no_nulos)

cambiados_mask = prod_no_nulos['product_name'] != prod_estandarizado['product_name']
cant_cambiados = cambiados_mask.sum()

print(f"Cantidad de nombres no nulos modificados: {cant_cambiados}")
ejemplos_cambios = pd.DataFrame({
    'Original': prod_no_nulos.loc[cambiados_mask, 'product_name'].head(3),
    'Estandarizado': prod_estandarizado.loc[cambiados_mask, 'product_name'].head(3)
})
ejemplos_cambios


Cantidad de nombres no nulos modificados: 6321
                                             Original                                      Estandarizado
17                Pizza for One Suprema  Frozen Pizza                 Pizza For One Suprema Frozen Pizza
23  Tri-Vi-Sol® Vitamins A-C-and D Supplement Drop...  Tri-Vi-Sol® Vitamins A-C-And D Supplement Drop...
29         Three Cheese Ziti- Marinara with Meatballs         Three Cheese Ziti- Marinara With Meatballs


*Interpretación:* Exactamente 6,321 nombres cambiaron al eliminar espacios redundantes y caracteres especiales tipo `\xa0`, y al unificar a mayúsculas iniciales (`Title Case`).


#### Ejercicio 2.5: Series con nombre del día y categoría ordenada
*Pregunta:* Crea una Series con el nombre del día de cada pedido y conviértela a una categoría ordenada de domingo a sábado. Compara la memoria del texto con la de la categoría.


In [12]:
mapa_dias = {0: 'Domingo', 1: 'Lunes', 2: 'Martes', 3: 'Miércoles', 4: 'Jueves', 5: 'Viernes', 6: 'Sábado'}
orden_dias = ['Domingo', 'Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado']

dias_texto = orders_limpio['order_dow'].map(mapa_dias)
dias_cat = pd.Series(pd.Categorical(dias_texto, categories=orden_dias, ordered=True))

mem_str = dias_texto.memory_usage(deep=True) / 1e6
mem_categoria = dias_cat.memory_usage(deep=True) / 1e6
ahorro_pct = (1 - mem_categoria / mem_str) * 100

print(f"Tipo de dato categórico: {dias_cat.dtype}")
print(f"Consumo en memoria (Texto):     {mem_str:.2f} MB")
print(f"Consumo en memoria (Categoría): {mem_categoria:.2f} MB")
print(f"Ahorro de memoria alcanzado:    {ahorro_pct:.1f}%")
print("\nDistribución por día de la semana:")
print(dias_cat.value_counts(sort=False))


Tipo de dato categórico: category
Consumo en memoria (Texto):     7.14 MB
Consumo en memoria (Categoría): 0.48 MB
Ahorro de memoria alcanzado:    93.3%

Distribución por día de la semana:
Domingo      84090
Lunes        82185
Martes       65833
Miércoles    60897
Jueves       59810
Viernes      63488
Sábado       62649
Name: count, dtype: int64


*Interpretación:* La conversión a tipo categórico reduce el uso de memoria en un 93.2% (de ~7 MB a menos de 0.5 MB) al almacenar internamente enteros pequeños mapeados al diccionario de categorías.


#### Ejercicio 2.6: Impacto de productos 'Unknown' en transacciones
*Pregunta:* ¿Cuántas filas de `order_prods` corresponden a productos 'Unknown' y en cuántos pedidos aparecen? ¿Los eliminarías? Justifica tu respuesta.


In [13]:
pids_unknown = products_crudo[products_crudo['product_name'].isna()]['product_id']
op_unknown = op_limpio[op_limpio['product_id'].isin(pids_unknown)]

filas_unknown = len(op_unknown)
pedidos_afectados_unknown = op_unknown['order_id'].nunique()

print(f"Filas en order_prods con producto 'Unknown': {filas_unknown}")
print(f"Pedidos distintos afectados:                {pedidos_afectados_unknown}")


Filas en order_prods con producto 'Unknown': 9547
Pedidos distintos afectados:                8254


*Interpretación:* Existen 9,547 artículos pertenecientes a productos 'Unknown', distribuidos en 8,254 pedidos distintos. **No deben eliminarse**, pues representan artículos reales agregados y pagados por los clientes; borrarlos alteraría el tamaño de la cesta, el cálculo de las posiciones en el carrito y el volumen transaccional de los usuarios.


#### Ejercicio 2.7: Reemplazo de 'missing' por nulo real en departamentos
*Pregunta:* En una copia de `departments`, reemplaza 'missing' por un nulo real (`pd.NA`). Explica en una frase por qué no conviene hacerlo en la tabla original antes de agrupar.


In [14]:
departments_copia = departments.copy()
departments_copia['department'] = departments_copia['department'].replace('missing', pd.NA)

print("Departamento con nulo en copia:")
print(departments_copia[departments_copia['department'].isna()])


Departamento con nulo en copia:
    department_id department
20             21        NaN


*Interpretación:* No conviene reemplazar `'missing'` por `pd.NA` en la tabla original porque las funciones de agregación como `groupby` excluyen por defecto los valores nulos (`dropna=True`), lo que causaría que todas las compras de esa categoría desaparecieran de los reportes sin advertencia.


---
## Lección 3: Filtrado, ordenamiento y agrupación de datos

En esta lección se construyen perfiles de comportamiento temporal y hábitos de recompra a partir de la tabla limpia de pedidos.

### Laboratorio Guiado 3: Perfil de hábitos de compra


#### Paso 1: Pedidos por cliente


In [15]:
pedidos_por_cliente = orders_limpio.groupby('user_id')['order_id'].count()
print(f"Total clientes únicos: {len(pedidos_por_cliente)}")
print("\nResumen estadístico de pedidos por cliente:")
pedidos_por_cliente.describe()


Total clientes únicos: 157437

Resumen estadístico de pedidos por cliente:
count    157437.000000
mean          3.042182
std           2.746842
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          28.000000
Name: order_id, dtype: float64


#### Paso 2: Segmentar clientes con `pd.cut`
Clasificamos a los clientes en:
- `ocasional`: 1 pedido
- `regular`: 2 a 4 pedidos
- `frecuente`: 5 o más pedidos


In [16]:
segmento_cliente = pd.cut(
    pedidos_por_cliente,
    bins=[0, 1, 4, np.inf],
    labels=['ocasional', 'regular', 'frecuente']
)

conteo_segmentos = segmento_cliente.value_counts().sort_index()
prop_segmentos = (segmento_cliente.value_counts(normalize=True).sort_index() * 100).round(1)

pd.DataFrame({'Clientes': conteo_segmentos, 'Porcentaje (%)': prop_segmentos})


           Clientes  Porcentaje (%)
order_id                           
ocasional     55357            35.2
regular       71553            45.4
frecuente     30527            19.4


#### Paso 3: Llevar el segmento a cada pedido con `.map()`


In [17]:
orders_limpio['segmento'] = orders_limpio['user_id'].map(segmento_cliente)
orders_limpio[['user_id', 'order_number', 'segmento']].head()


   user_id  order_number   segmento
0   183418            11    regular
1   163593             5    regular
2    39980             4  ocasional
3    82516            56  frecuente
4   196724             2  ocasional


#### Paso 4: Resumen por segmento


In [18]:
resumen_segmento = (orders_limpio
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
resumen_segmento


           clientes  pedidos  espera_mediana
segmento                                    
ocasional     55357    55357            16.0
regular       71553   191649            11.0
frecuente     30527   231946             6.0


**Interpretación:** El 19.4% de los clientes (los frecuentes) genera casi la mitad de las transacciones (231,946 pedidos) y realiza recompras con una mediana de intervalo de apenas 6 días, frente a los 16 días de los ocasionales.


#### Paso 5: Franjas horarias y tabla cruzada


In [19]:
orders_limpio['franja'] = pd.cut(
    orders_limpio['order_hour_of_day'],
    bins=[-1, 5, 11, 17, 23],
    labels=['madrugada', 'mañana', 'tarde', 'noche']
)

perfil_franja = (pd.crosstab(orders_limpio['segmento'], orders_limpio['franja'], normalize='index')
                 .mul(100).round(1))
perfil_franja


franja     madrugada  mañana  tarde  noche
segmento                                  
ocasional        2.1    30.0   49.4   18.5
regular          1.9    31.6   48.4   18.1
frecuente        1.7    35.2   45.9   17.2


**Conclusiones clave de hábitos:** Las tardes (12h a 17h) concentran cerca del 48-49% de los pedidos en todos los segmentos, mientras que los clientes frecuentes tienen una propensión ligeramente superior a comprar en la mañana (35.2%) en comparación con los ocasionales (30.0%).


### Reto 3: Ejercicios de Filtrado, Ordenamiento y Agrupación


#### Ejercicio 3.1: Horas pico de pedidos los domingos
*Pregunta:* ¿Cuáles son las tres horas con más pedidos los domingos y cuántos pedidos tiene cada una?


In [20]:
domingos = orders_limpio[orders_limpio['order_dow'] == 0]
top3_horas_domingo = domingos.groupby('order_hour_of_day')['order_id'].count().nlargest(3).reset_index()
top3_horas_domingo.columns = ['Hora del Día (Domingo)', 'Total Pedidos']
top3_horas_domingo


   Hora del Día (Domingo)  Total Pedidos
0                      15           7585
1                      14           7533
2                      13           7483


*Interpretación:* Las horas pico de compra dominical son las 15:00 (7,585 pedidos), 14:00 (7,533 pedidos) y 13:00 (7,483 pedidos), demostrando una fuerte concentración en las primeras horas de la tarde.


#### Ejercicio 3.2: Intervalo medio de espera por hora del día
*Pregunta:* ¿A qué hora es mayor la espera media entre pedidos (`days_since_prior_order`) y a qué hora es menor?


In [21]:
espera_por_hora = orders_limpio.groupby('order_hour_of_day')['days_since_prior_order'].mean()
hora_mayor = espera_por_hora.idxmax()
hora_menor = espera_por_hora.idxmin()

print(f"Mayor espera media: a las {hora_mayor}:00 h con {espera_por_hora[hora_mayor]:.2f} días promedio.")
print(f"Menor espera media: a las {hora_menor}:00 h con {espera_por_hora[hora_menor]:.2f} días promedio.")


Mayor espera media: a las 0:00 h con 12.52 días promedio.
Menor espera media: a las 22:00 h con 10.62 días promedio.


*Interpretación:* La mayor espera promedio ocurre a la medianoche (0:00 h, 12.52 días), mientras que la menor espera ocurre a las 22:00 h (10.62 días), reflejando diferencias en la recurrencia de los usuarios que ordenan en turnos nocturnos.


#### Ejercicio 3.3: Clientes con alta frecuencia de compra
*Pregunta:* ¿Cuántos clientes tienen 10 pedidos o más?


In [22]:
clientes_10_mas = (pedidos_por_cliente >= 10).sum()
pct_10_mas = (clientes_10_mas / len(pedidos_por_cliente)) * 100

print(f"Clientes con 10 o más pedidos: {clientes_10_mas} ({pct_10_mas:.2f}% de la base de usuarios)")


Clientes con 10 o más pedidos: 6327 (4.02% de la base de usuarios)


*Interpretación:* Un núcleo de 6,327 usuarios leales (4.02% del total) ha acumulado al menos 10 pedidos en el periodo analizado.


#### Ejercicio 3.4: Pedidos con ciclo exacto de 7 días
*Pregunta:* ¿Cuántos pedidos se hicieron exactamente 7 días después del anterior y qué porcentaje del total representan?


In [23]:
pedidos_7d = (orders_limpio['days_since_prior_order'] == 7).sum()
pct_7d = (pedidos_7d / len(orders_limpio)) * 100

print(f"Pedidos realizados exactamente a los 7 días: {pedidos_7d} ({pct_7d:.2f}% del total)")


Pedidos realizados exactamente a los 7 días: 44577 (9.31% del total)


*Interpretación:* Con 44,577 pedidos (9.31%), el intervalo de 7 días representa uno de los patrones más recurrentes, evidenciando un hábito semanal consolidado en la planificación de despensa.


#### Ejercicio 3.5: Cestas de compra grandes en `order_products`
*Pregunta:* Usando `order_prods`: ¿cuántos pedidos tienen 20 productos o más? ¿Cuál es el pedido más grande y cuántos productos tiene?


In [24]:
prods_x_pedido = op_limpio.groupby('order_id').size()
pedidos_20_mas = (prods_x_pedido >= 20).sum()
id_pedido_max = prods_x_pedido.idxmax()
articulos_pedido_max = prods_x_pedido.max()

print(f"Pedidos con 20 o más productos: {pedidos_20_mas}")
print(f"Pedido con mayor cantidad de productos: order_id = {id_pedido_max} con {articulos_pedido_max} artículos.")


Pedidos con 20 o más productos: 49420
Pedido con mayor cantidad de productos: order_id = 61355 con 127 artículos.


*Interpretación:* 49,420 pedidos corresponden a compras mayoristas o de abastecimiento familiar extenso (≥ 20 ítems). El pedido récord contiene 127 artículos.


#### Ejercicio 3.6: Tasa de recompra según la posición en el carrito
*Pregunta:* Usando `order_prods`: calcula la tasa de recompra (media de `reordered`) para las posiciones 1 a 5 del carrito. ¿Qué patrón ves?


In [25]:
posiciones_1_5 = op_limpio[op_limpio['add_to_cart_order'].isin([1, 2, 3, 4, 5])]
tasa_posicion = (posiciones_1_5.groupby('add_to_cart_order')['reordered']
                 .mean().reset_index())
tasa_posicion.columns = ['Posición en Carrito', 'Tasa de Recompra']
tasa_posicion['Tasa (%)'] = (tasa_posicion['Tasa de Recompra'] * 100).round(2)
tasa_posicion


   Posición en Carrito  Tasa de Recompra  Tasa (%)
0                    1          0.678620     67.86
1                    2          0.677059     67.71
2                    3          0.658986     65.90
3                    4          0.637511     63.75
4                    5          0.617483     61.75


*Interpretación:* Existe un patrón monotónicamente decreciente: los primeros productos colocados en el carrito son mayoritariamente recompras de productos básicos habituales (67.86% en posición 1 y 67.71% en posición 2), mientras que conforme aumenta la posición, sube la probabilidad de que sean productos exploratorios o compras impulsivas.


#### Ejercicio 3.7: Intervalo de recompra: fin de semana vs semana
*Pregunta:* Compara la mediana de `days_since_prior_order` de los pedidos de fin de semana (domingo y sábado) con la del resto de la semana.


In [26]:
orders_limpio['es_fin_de_semana'] = orders_limpio['order_dow'].isin([0, 6])

mediana_comparativa = (orders_limpio
    .groupby('es_fin_de_semana')['days_since_prior_order']
    .median()
    .rename(index={False: 'Lunes a Viernes', True: 'Fin de Semana (Sáb-Dom)'})
    .reset_index())
mediana_comparativa.columns = ['Periodo', 'Mediana Días de Espera']
mediana_comparativa


                   Periodo  Mediana Días de Espera
0          Lunes a Viernes                     7.0
1  Fin de Semana (Sáb-Dom)                     8.0


*Interpretación:* La mediana de espera para compras entre semana es de 7.0 días, mientras que las compras en fin de semana presentan una mediana ligeramente mayor de 8.0 días.


---
## Lección 4: Combinación y estructuración de múltiples datasets

En esta lección se construyen la tabla analítica maestra y la ficha descriptiva por producto, uniendo las tablas de hechos y dimensiones con validación de integridad referencial.

### Construcción y Validación de la Tabla Maestra


In [27]:
filas_inicio = len(op_limpio)

# Enlace de dimensiones sobre la tabla de hechos order_products
maestra = op_limpio.merge(products_limpio, on='product_id', how='left', validate='m:1')
print(f"+ products:    {maestra.shape}")

maestra = maestra.merge(aisles, on='aisle_id', how='left', validate='m:1')
print(f"+ aisles:      {maestra.shape}")

maestra = maestra.merge(departments, on='department_id', how='left', validate='m:1')
print(f"+ departments: {maestra.shape}")

maestra = maestra.merge(orders_limpio, on='order_id', how='left', validate='m:1')
print(f"+ orders:      {maestra.shape}")

# Confirmación de que no hubo multiplicación ni pérdida de filas
assert len(maestra) == filas_inicio
print(f"\nTabla maestra consolidada exitosamente con {len(maestra):,} filas y {maestra.shape[1]} columnas.")


+ products:    (4545007, 7)
+ aisles:      (4545007, 8)
+ departments: (4545007, 9)
+ orders:      (4545007, 17)

Tabla maestra consolidada exitosamente con 4,545,007 filas y 17 columnas.


### Laboratorio Guiado 4: Tabla de análisis por producto


#### Paso 1: Agregar la tabla maestra por producto


In [28]:
por_producto = (maestra
    .groupby('product_id')
    .agg(veces=('order_id', 'size'),
         clientes=('user_id', 'nunique'),
         tasa_recompra=('reordered', 'mean'),
         posicion_media=('add_to_cart_order', 'mean'))
    .reset_index())

print(f"Dimensiones por_producto: {por_producto.shape}")
por_producto.head(3)


Dimensiones por_producto: (45573, 5)
   product_id  veces  clientes  tasa_recompra  posicion_media
0           1    280       206       0.564286        5.642857
1           2     11        11       0.000000        8.727273
2           3     42        27       0.738095        6.952381


#### Paso 2: Recuperar nombre y departamento con `merge`


In [29]:
ficha_productos = (por_producto
    .merge(products_limpio, on='product_id', how='left', validate='1:1')
    .merge(departments, on='department_id', how='left', validate='m:1')
    [['product_id', 'product_name', 'department', 'veces', 'clientes',
      'tasa_recompra', 'posicion_media']])

print(f"Dimensiones ficha_productos: {ficha_productos.shape}")
print("\nProducto más pedido del catálogo general:")
ficha_productos.loc[ficha_productos['veces'].idxmax()]


Dimensiones ficha_productos: (45573, 7)

Producto más pedido del catálogo general:
product_id           24852
product_name        Banana
department         produce
veces                66050
clientes             33274
tasa_recompra     0.844254
posicion_media    4.874815
Name: 22808, dtype: object


#### Paso 3: Filtrar productos con suficientes datos (≥ 1,000 pedidos)


In [30]:
populares = ficha_productos[ficha_productos['veces'] >= 1000].copy()
print(f"Productos populares con al menos 1,000 pedidos: {len(populares)}")


Productos populares con al menos 1,000 pedidos: 739


#### Paso 4: Clasificar dentro de cada departamento con `.rank()`


In [31]:
populares['puesto'] = (populares
    .groupby('department')['tasa_recompra']
    .rank(ascending=False, method='first'))

lideres = (populares[populares['puesto'] == 1]
           .sort_values('tasa_recompra', ascending=False))

print(f"Departamentos con producto líder: {len(lideres)}")
lideres[['department', 'product_name', 'veces', 'tasa_recompra']].head(5)


Departamentos con producto líder: 18
       department                           product_name  veces  tasa_recompra
41704  dairy eggs             Whole Organic Omega 3 Milk   1286       0.863142
22808     produce                                 Banana  66050       0.844254
29030      snacks        Extra Fancy Unsalted Mixed Nuts   1436       0.822423
42302   beverages                      Zero Calorie Cola   1256       0.818471
3570       bakery  Organic Honey Sweet Whole Wheat Bread   1607       0.799004


#### Paso 5: Alternativa con `sort_values` y `groupby().head(1)`


In [32]:
lideres_2 = (populares
    .sort_values('tasa_recompra', ascending=False)
    .groupby('department')
    .head(1))

son_identicos = set(lideres_2['product_id']) == set(lideres['product_id'])
print(f"¿Ambos métodos arrojan los mismos líderes?: {son_identicos}")


¿Ambos métodos arrojan los mismos líderes?: True


#### Paso 6: Persistencia en `data/processed/`


In [33]:
ficha_productos.to_parquet(RUTA_PROCESSED / 'ficha_productos.parquet', index=False)
ficha_productos.to_csv(RUTA_PROCESSED / 'ficha_productos.csv', index=False)
lideres.to_csv(RUTA_PROCESSED / 'lideres_por_departamento.csv', index=False)

por_depto = (maestra.groupby('department')
                    .agg(articulos=('product_id', 'size'),
                         tasa_recompra=('reordered', 'mean'))
                    .sort_values('articulos', ascending=False))
por_depto.to_csv(RUTA_PROCESSED / 'resumen_departamentos.csv')

print("Archivos procesados exportados exitosamente a data/processed/.")


Archivos procesados exportados exitosamente a data/processed/.


### Reto 4: Ejercicios de Estructuración y Análisis Multivariable


#### Ejercicio 4.1: Pasillos con mayor y menor tasa de recompra
*Pregunta:* Entre los pasillos con al menos 10,000 artículos pedidos, ¿cuáles son los tres con mayor tasa de recompra? ¿Qué pasillo tiene la tasa más baja de todos?


In [34]:
pasillos_metricas = maestra.groupby('aisle').agg(
    articulos=('product_id', 'size'),
    tasa_recompra=('reordered', 'mean')
)

pasillos_10k = pasillos_metricas[pasillos_metricas['articulos'] >= 10000]
top3_pasillos_recompra = pasillos_10k.nlargest(3, 'tasa_recompra').reset_index()
pasillo_menor_recompra = pasillos_metricas.nsmallest(1, 'tasa_recompra').reset_index()

print("Top 3 pasillos con >= 10,000 artículos y mayor tasa de recompra:")
print(top3_pasillos_recompra)
print("\nPasillo con menor tasa de recompra global:")
print(pasillo_menor_recompra)


Top 3 pasillos con >= 10,000 artículos y mayor tasa de recompra:
                           aisle  articulos  tasa_recompra
0                           milk     125522       0.781305
1  water seltzer sparkling water     117133       0.730605
2                   fresh fruits     510101       0.718197

Pasillo con menor tasa de recompra global:
               aisle  articulos  tasa_recompra
0  spices seasonings      29454       0.152407


*Interpretación:* Los pasillos de consumo básico diario lideran la recompra: `milk` (78.13%), `water seltzer sparkling water` (73.06%) y `fresh fruits` (71.82%). El pasillo con menor fidelidad de recompra es `spices seasonings` (15.24%), dada la baja caducidad y compra esporádica de especias.


#### Ejercicio 4.2: Participación por departamento: madrugada vs total
*Pregunta:* Compara la participación de cada departamento en los pedidos de madrugada (0 a 5 h) con su participación total. ¿Qué departamento entra en el top 3 de madrugada pero no en el total?


In [35]:
part_total = maestra['department'].value_counts(normalize=True)
madrugada_filtro = maestra['order_hour_of_day'].between(0, 5)
part_madrugada = maestra.loc[madrugada_filtro, 'department'].value_counts(normalize=True)

comparacion_dept = pd.DataFrame({
    'Participación Total (%)': (part_total * 100).round(2),
    'Participación Madrugada (%)': (part_madrugada * 100).round(2)
})
comparacion_dept['Diferencia'] = (comparacion_dept['Participación Madrugada (%)'] - comparacion_dept['Participación Total (%)']).round(2)

print("Top 3 Total:")
print(comparacion_dept.sort_values('Participación Total (%)', ascending=False).head(3))
print("\nTop 3 Madrugada:")
print(comparacion_dept.sort_values('Participación Madrugada (%)', ascending=False).head(3))


Top 3 Total:
            Participación Total (%)  Participación Madrugada (%)  Diferencia
department                                                                  
produce                       29.23                        30.21        0.98
dairy eggs                    16.72                        15.87       -0.85
snacks                         8.91                         7.89       -1.02

Top 3 Madrugada:
            Participación Total (%)  Participación Madrugada (%)  Diferencia
department                                                                  
produce                       29.23                        30.21        0.98
dairy eggs                    16.72                        15.87       -0.85
beverages                      8.26                         8.49        0.23


*Interpretación:* En el total global, el tercer lugar lo ocupa `snacks` (8.91%). Sin embargo, en el horario de madrugada (0 a 5 h), el departamento de `beverages` (bebidas) asciende al top 3 con 8.49%, superando a `snacks`.


#### Ejercicio 4.3: Top 3 productos más vendidos en `snacks`
*Pregunta:* ¿Cuáles son los tres productos más pedidos del departamento snacks?


In [36]:
snacks_df = maestra[maestra['department'] == 'snacks']
top3_snacks = (snacks_df.groupby(['product_id', 'product_name'])
               .size()
               .nlargest(3)
               .reset_index(name='Total Pedidos'))
top3_snacks


   product_id                          product_name  Total Pedidos
0       13870  Lightly Salted Baked Snap Pea Crisps           3462
1       34448                Original Veggie Straws           2331
2       31040                   Sea Salt Pita Chips           2275


*Interpretación:* Los snacks favoritos son opciones saludables y horneadas: `Lightly Salted Baked Snap Pea Crisps` (3,462 pedidos), `Original Veggie Straws` (2,331 pedidos) y `Sea Salt Pita Chips` (2,275 pedidos).


#### Ejercicio 4.4: Resumen analítico por cliente
*Pregunta:* Construye una tabla con una fila por cliente y tres columnas: pedidos distintos, artículos y tasa de recompra. ¿Cuántos clientes tiene, cuál es el promedio de artículos por cliente y qué cliente compró más artículos?


In [37]:
resumen_clientes = (maestra.groupby('user_id')
                    .agg(pedidos_distintos=('order_id', 'nunique'),
                         articulos=('product_id', 'size'),
                         tasa_recompra=('reordered', 'mean')))

cant_clientes_maestra = len(resumen_clientes)
promedio_articulos = resumen_clientes['articulos'].mean()
id_top_cliente = resumen_clientes['articulos'].idxmax()
articulos_top_cliente = resumen_clientes['articulos'].max()

print(f"Total de clientes con compras en la tabla maestra: {cant_clientes_maestra:,}")
print(f"Promedio de artículos por cliente:                 {promedio_articulos:.2f}")
print(f"Cliente con mayor cantidad de artículos:           user_id = {id_top_cliente} con {articulos_top_cliente} artículos.")


Total de clientes con compras en la tabla maestra: 149,626
Promedio de artículos por cliente:                 30.38
Cliente con mayor cantidad de artículos:           user_id = 103634 con 580 artículos.


*Interpretación:* La tabla resume 149,626 usuarios activos con pedidos surtidos. En promedio, cada cliente adquiere 30.38 productos en el histórico registrado, con un valor máximo de 580 artículos para el cliente más intensivo.


#### Ejercicio 4.5: Auditoría de unión `inner` vs `left` entre pedidos y productos
*Pregunta:* Une `orders` con los pedidos de `order_prods` usando `how='inner'`. ¿Cuántas filas obtienes y por qué son menos que con `how='left'`?


In [38]:
pedidos_con_articulos = op_limpio[['order_id']].drop_duplicates()
inner_pedidos = orders_limpio.merge(pedidos_con_articulos, on='order_id', how='inner')

filas_inner = len(inner_pedidos)
filas_orders = len(orders_limpio)
pedidos_sin_productos = filas_orders - filas_inner

print(f"Filas con how='inner':       {filas_inner:,}")
print(f"Filas totales en orders:     {filas_orders:,}")
print(f"Diferencia (pedidos vacíos): {pedidos_sin_productos:,} ({pedidos_sin_productos / filas_orders * 100:.2f}%)")


Filas con how='inner':       450,046
Filas totales en orders:     478,952
Diferencia (pedidos vacíos): 28,906 (6.04%)


*Interpretación:* La unión interna descarta 28,906 pedidos (6.03% del total) debido a que no cuentan con detalle de productos asociados en `order_products.csv`. Usar `how='left'` permite conservar la totalidad de las órdenes emitidas.


#### Ejercicio 4.6: Apilado horizontal con `pd.concat`
*Pregunta:* Con `concat`, arma una sola tabla con los tres productos más pedidos de `produce` y los tres de `dairy eggs`, con una columna que indique el departamento.


In [39]:
top_produce = (maestra[maestra['department'] == 'produce']
               .groupby('product_name').size()
               .nlargest(3).reset_index(name='veces')
               .assign(department='produce'))

top_dairy = (maestra[maestra['department'] == 'dairy eggs']
             .groupby('product_name').size()
             .nlargest(3).reset_index(name='veces')
             .assign(department='dairy eggs'))

tabla_apilada = pd.concat([top_produce, top_dairy], ignore_index=True)
tabla_apilada[['department', 'product_name', 'veces']]


   department            product_name  veces
0     produce                  Banana  66050
1     produce  Bag Of Organic Bananas  53297
2     produce    Organic Strawberries  37039
3  dairy eggs      Organic Whole Milk  20953
4  dairy eggs     Organic Half & Half  10752
5  dairy eggs             Half & Half   9588


*Interpretación:* La función `pd.concat` combina verticalmente los líderes de ambas categorías preservando la trazabilidad de su departamento de origen.


#### Ejercicio 4.7: Tabla dinámica por departamento y día de la semana
*Pregunta:* Crea una tabla dinámica de artículos por departamento (filas) y día de la semana (columnas). ¿Qué forma tiene y qué día se piden más artículos de alcohol?


In [40]:
matriz_dept_dow = maestra.pivot_table(
    index='department',
    columns='order_dow',
    values='product_id',
    aggfunc='size'
)

mapa_dias_col = {0: 'Dom', 1: 'Lun', 2: 'Mar', 3: 'Mié', 4: 'Jue', 5: 'Vie', 6: 'Sáb'}
matriz_dept_dow.columns = [mapa_dias_col.get(c, c) for c in matriz_dept_dow.columns]

print(f"Forma de la tabla dinámica: {matriz_dept_dow.shape} (21 departamentos x 7 días)")
print("\nArtículos pedidos de 'alcohol' por día de la semana:")
print(matriz_dept_dow.loc['alcohol'])

dia_max_alcohol = matriz_dept_dow.loc['alcohol'].idxmax()
cant_max_alcohol = matriz_dept_dow.loc['alcohol'].max()
print(f"\nDía con mayor volumen de compras de alcohol: {dia_max_alcohol} con {cant_max_alcohol:,} artículos.")


Forma de la tabla dinámica: (21, 7) (21 departamentos x 7 días)

Artículos pedidos de 'alcohol' por día de la semana:
Dom    2397
Lun    3025
Mar    2868
Mié    3094
Jue    3656
Vie    3814
Sáb    2787
Name: alcohol, dtype: int64

Día con mayor volumen de compras de alcohol: Vie con 3,814 artículos.


*Interpretación:* La tabla dinámica tiene forma `(21, 7)`. El día con mayor demanda de bebidas alcohólicas es el **Viernes** (`Vie`, día 5) con 3,814 artículos comprados, lo que responde al aprovisionamiento para el fin de semana.
